# AIC2024 KIS — Phase 2: CLIP embeddings theo từng clip (Kaggle)

Notebook **self-contained**: chỉ cần bundle keyframe, không cần SQLite.

## Chuẩn bị
1. Ở local tạo bundle: `uv run ir export-keyframes`
   → `artifacts_keyframes.zip` (gồm `keyframes/` + `manifest.jsonl`).
2. Upload zip đó thành Kaggle Dataset.
3. Notebook Settings: bật **GPU** và **Internet**.
4. Add dataset, chọn batch (bên dưới), chạy hết cell.
5. Tải `artifacts_embed_shards.zip` về máy rồi import:
   `uv run ir import-shards artifacts_embed_shards.zip`
   (tự giải nén vào `state/embed/shards/` và merge FAISS).

## Chọn batch
- Mặc định: tất cả video trong manifest.
- Theo lô: `EMBED_VIDEO_OFFSET` + `EMBED_VIDEO_LIMIT` (ví dụ 0/100, 100/100, ...).
- Hoặc chỉ định: `EMBED_VIDEO_IDS=L01_V001,L01_V002`.
- Resume: shard đã xong trong `/kaggle/working` sẽ được bỏ qua.

## Output mỗi shard
`shards/<video_id>/embeddings.f16.npy`, `id_map.npy`, `manifest.json`.


## 1. Cấu hình

In [ ]:
import os
from pathlib import Path

MODEL_NAME = "sentence-transformers/clip-ViT-B-16"
DIM = 512
BATCH_SIZE = 128
NUM_WORKERS = int(os.environ.get("EMBED_NUM_WORKERS", "4"))
PROGRESS_EVERY = int(os.environ.get("EMBED_PROGRESS_EVERY", "20"))

BUNDLE_DIR = os.environ.get("EMBED_BUNDLE_DIR") or None
WORK_DIR = Path(os.environ.get("EMBED_WORK_DIR", "/kaggle/working/embed"))
SHARDS_DIR = Path(os.environ.get("EMBED_SHARDS_DIR", str(WORK_DIR / "shards")))
ZIP_PATH = Path(os.environ.get("EMBED_ZIP_PATH", "/kaggle/working/artifacts_embed_shards.zip"))
RESUME = os.environ.get("EMBED_RESUME", "1") != "0"

VIDEO_IDS = [
    value.strip()
    for value in os.environ.get("EMBED_VIDEO_IDS", "").split(",")
    if value.strip()
]
VIDEO_OFFSET = int(os.environ.get("EMBED_VIDEO_OFFSET", "0"))
VIDEO_LIMIT = int(os.environ.get("EMBED_VIDEO_LIMIT", "0"))

print("model:", MODEL_NAME, "| dim:", DIM, "| batch:", BATCH_SIZE)
print("decode threads:", NUM_WORKERS, "| progress every:", PROGRESS_EVERY, "batches")
print("bundle override:", BUNDLE_DIR)
print("work:", WORK_DIR)
print("shards:", SHARDS_DIR)
print("zip:", ZIP_PATH)
print("video_ids:", VIDEO_IDS or "(all)", "| offset:", VIDEO_OFFSET, "| limit:", VIDEO_LIMIT)


## 2. Cài dependency

Kaggle đã có `torch`; cell này bổ sung `sentence-transformers` và `faiss-cpu`.

In [ ]:
import subprocess
import sys

if os.environ.get("EMBED_SKIP_INSTALL") != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "sentence-transformers", "faiss-cpu"],
        check=True,
    )
else:
    print("skip install (EMBED_SKIP_INSTALL=1)")


## 3. Tìm bundle keyframe

Tìm `manifest.jsonl` có thư mục `keyframes/` đi kèm trong `/kaggle/input`.
Manifest chứa `kf_uid`, đường dẫn ảnh và metadata; không cần đọc SQLite.

In [ ]:
import json

candidates = []
seen = set()


def add_bundle(candidate: Path) -> None:
    bundle = candidate.resolve()
    if bundle in seen:
        return
    seen.add(bundle)
    if (bundle / "manifest.jsonl").is_file() and (bundle / "keyframes").is_dir():
        candidates.append(bundle)


if BUNDLE_DIR:
    # Đường dẫn đã chỉ định thì dùng thẳng. rglob trên /kaggle/input (FUSE mount,
    # 280k file) tốn nhiều phút mà không tìm được bundle nào khác.
    add_bundle(Path(BUNDLE_DIR))
else:
    for root in (Path("/kaggle/input"), Path.cwd()):
        if not root.is_dir():
            continue
        for manifest in sorted(root.rglob("manifest.jsonl")):
            add_bundle(manifest.parent)

if not candidates:
    raise FileNotFoundError(
        "Không tìm thấy manifest.jsonl + keyframes/. "
        "Set EMBED_BUNDLE_DIR hoặc kiểm tra lại Kaggle dataset."
    )

BUNDLE_DIR = candidates[0]
MANIFEST_PATH = BUNDLE_DIR / "manifest.jsonl"

rows = []
for line in MANIFEST_PATH.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line:
        rows.append(json.loads(line))
if not rows:
    raise RuntimeError("manifest.jsonl rỗng")

print("bundle:", BUNDLE_DIR)
print("keyframes:", BUNDLE_DIR / "keyframes")
print("manifest rows:", f"{len(rows):,}")
if len(candidates) > 1:
    print("bỏ qua bundle khác:", [str(path) for path in candidates[1:]])


## 4. Chọn batch video

Thứ tự video lấy theo `video_id` đã sort. `EMBED_VIDEO_OFFSET`/`EMBED_VIDEO_LIMIT`
cho phép chạy lô 100 video mỗi lần mà không cần sửa notebook.

In [ ]:
import numpy as np

by_video: dict[str, list[dict]] = {}
for row in sorted(rows, key=lambda item: (str(item["video_id"]), int(item["kf_uid"]))):
    by_video.setdefault(str(row["video_id"]), []).append(row)

all_video_ids = sorted(by_video)
if VIDEO_IDS:
    selected = list(dict.fromkeys(VIDEO_IDS))
    missing_videos = [video_id for video_id in selected if video_id not in by_video]
    if missing_videos:
        raise ValueError(f"EMBED_VIDEO_IDS không có trong manifest: {missing_videos}")
elif VIDEO_LIMIT > 0:
    selected = all_video_ids[VIDEO_OFFSET : VIDEO_OFFSET + VIDEO_LIMIT]
else:
    selected = all_video_ids[VIDEO_OFFSET:]

if not selected:
    raise RuntimeError("Không có video nào được chọn.")

missing_files = []
total_selected = sum(len(by_video[video_id]) for video_id in selected)

# Kiểm tra từng file tốn 280k lần stat() trên FUSE mount (10-20 phút) và
# manifest vốn đã liệt kê chính xác file cần có. Cell embed vẫn mở ảnh thật
# nên file thiếu sẽ lỗi ngay tại đó. Bật EMBED_VERIFY_FILES=1 để kiểm tra trước.
if os.environ.get("EMBED_VERIFY_FILES") == "1":
    for video_id in selected:
        for row in by_video[video_id]:
            image_path = BUNDLE_DIR / row["path"]
            try:
                if image_path.stat().st_size == 0:
                    missing_files.append(str(image_path))
            except OSError:
                missing_files.append(str(image_path))

print(f"videos in manifest: {len(all_video_ids):,}")
print(f"selected videos: {len(selected):,} | keyframes: {total_selected:,}")
print(f"batch: offset={VIDEO_OFFSET} limit={VIDEO_LIMIT} explicit={len(VIDEO_IDS)}")
if selected:
    print("first:", selected[0], "| last:", selected[-1])
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next EMBED_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))
if os.environ.get("EMBED_VERIFY_FILES") == "1":
    print(f"missing/empty: {len(missing_files):,}")
    if missing_files:
        raise FileNotFoundError("Thiếu file keyframe:\n" + "\n".join(missing_files[:20]))


## 5. Load CLIP ViT-B/16

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = DEVICE == "cuda"
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    print("device:", DEVICE, "|", torch.cuda.get_device_name(0), "| fp16:", USE_FP16)
else:
    print("device:", DEVICE)
    if os.environ.get("EMBED_REQUIRE_GPU", "1") != "0":
        raise RuntimeError(
            "GPU không khả dụng — bật Accelerator=GPU trong Kaggle Settings. "
            "Đặt EMBED_REQUIRE_GPU=0 để chạy CPU (rất chậm)."
        )

model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model_dim = (
    model.get_embedding_dimension()
    if hasattr(model, "get_embedding_dimension")
    else model.get_sentence_embedding_dimension()
)
if model_dim != DIM:
    raise RuntimeError(f"model dim {model_dim} != configured {DIM}")
print("embedding dim:", model_dim)


## 6. Embed từng video thành shard

Mỗi video ghi vào `SHARDS_DIR/<video_id>/`. Shard đã xong được bỏ qua khi resume;
shard dở dang đọc `progress.json` để chạy tiếp.

In [ ]:
import hashlib
import json
import shutil
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import UTC, datetime

import torch
from PIL import Image


def load_keyframe(row: dict) -> Image.Image:
    with Image.open(BUNDLE_DIR / row["path"]) as image:
        return image.convert("RGB")


def write_json(path: Path, payload: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    tmp = path.with_name(f"{path.stem}.tmp.npy")
    np.save(tmp, array)
    os.replace(tmp, path)


decode_pool = ThreadPoolExecutor(max_workers=NUM_WORKERS)
shard_summaries = []
overall_started = time.time()

for position, video_id in enumerate(selected, start=1):
    video_rows = by_video[video_id]
    total = len(video_rows)
    uids = np.asarray([int(row["kf_uid"]) for row in video_rows], dtype=np.int64)
    current_hash = hashlib.sha256(
        MODEL_NAME.encode() + b"\n" + uids.tobytes()
    ).hexdigest()

    shard_dir = SHARDS_DIR / video_id
    if not RESUME:
        shutil.rmtree(shard_dir, ignore_errors=True)
    shard_dir.mkdir(parents=True, exist_ok=True)

    embeddings_path = shard_dir / "embeddings.f16.npy"
    progress_path = shard_dir / "progress.json"
    plan_path = shard_dir / "plan.json"
    manifest_path = shard_dir / "manifest.json"

    rows_done = 0
    plan_ok = False
    if RESUME and embeddings_path.is_file() and plan_path.is_file():
        try:
            plan = json.loads(plan_path.read_text(encoding="utf-8"))
            probe = np.load(embeddings_path, mmap_mode="r")
            plan_ok = (
                plan.get("plan_hash") == current_hash
                and int(plan.get("count", -1)) == total
                and probe.shape == (total, DIM)
            )
            if plan_ok:
                progress = json.loads(progress_path.read_text(encoding="utf-8"))
                rows_done = min(max(0, int(progress.get("rows_done", 0))), total)
            del probe
        except (OSError, ValueError, KeyError):
            plan_ok = False

    if not plan_ok:
        embeddings_path.unlink(missing_ok=True)
        progress_path.unlink(missing_ok=True)
        plan_path.unlink(missing_ok=True)
        memmap = np.lib.format.open_memmap(
            str(embeddings_path), mode="w+", dtype=np.float16, shape=(total, DIM)
        )
        del memmap
        rows_done = 0

    write_json(
        plan_path,
        {
            "video_id": video_id,
            "model": MODEL_NAME,
            "dim": DIM,
            "count": total,
            "plan_hash": current_hash,
        },
    )

    if RESUME and rows_done == total and manifest_path.is_file():
        print(f"[{position}/{len(selected)}] {video_id}: skip (done)")
        shard_summaries.append({"video_id": video_id, "count": total, "skipped": True})
        continue

    write_json(progress_path, {"rows_done": rows_done, "status": "running"})
    started = time.time()
    memmap = np.lib.format.open_memmap(str(embeddings_path), mode="r+")

    def load_batch(start: int) -> list[Image.Image]:
        return [load_keyframe(row) for row in video_rows[start : start + BATCH_SIZE]]

    pending = decode_pool.submit(load_batch, rows_done) if rows_done < total else None
    for batch_index, start in enumerate(range(rows_done, total, BATCH_SIZE), start=1):
        images = pending.result()
        upcoming = start + BATCH_SIZE
        pending = decode_pool.submit(load_batch, upcoming) if upcoming < total else None

        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            vectors = model.encode(
                images,
                batch_size=len(images),
                normalize_embeddings=True,
                convert_to_numpy=True,
                show_progress_bar=False,
            )
        memmap[start : start + len(images)] = np.asarray(vectors, dtype=np.float16)

        if batch_index % PROGRESS_EVERY == 0:
            memmap.flush()
            write_json(
                progress_path,
                {"rows_done": start + len(images), "status": "running"},
            )

    memmap.flush()
    del memmap
    write_json(progress_path, {"rows_done": total, "status": "done"})

    atomic_save_npy(shard_dir / "id_map.npy", uids)
    manifest = {
        "version": 1,
        "video_id": video_id,
        "created_at": datetime.now(UTC).isoformat(),
        "model": MODEL_NAME,
        "dim": DIM,
        "count": total,
        "dtype": "float16",
        "normalized": "l2",
        "order": "kf_uid",
        "plan_hash": current_hash,
        "files": {"embeddings": "embeddings.f16.npy", "id_map": "id_map.npy"},
    }
    write_json(manifest_path, manifest)
    shard_summaries.append({"video_id": video_id, "count": total, "skipped": False})
    print(
        f"[{position}/{len(selected)}] {video_id}: {total:,} keyframes "
        f"in {time.time() - started:,.1f}s"
    )

print(
    f"done {len(shard_summaries):,} videos in "
    f"{(time.time() - overall_started) / 60:,.1f} min"
)


## 7. Đóng gói shard

Zip chỉ chứa shard của batch này (không chứa keyframes, không chứa file tạm).

In [ ]:
import zipfile

README = """AIC2024 KIS - per-video embedding shards
Import locally with:
  uv run ir import-shards artifacts_embed_shards.zip
This extracts to state/embed/shards/<video_id>/ and rebuilds the FAISS index.
"""

ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
written = 0
with zipfile.ZipFile(
    ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
) as archive:
    for summary in shard_summaries:
        shard_dir = SHARDS_DIR / summary["video_id"]
        for path in sorted(shard_dir.rglob("*")):
            if not path.is_file() or path.name in {"progress.json", "plan.json"}:
                continue
            archive.write(
                path,
                arcname=path.relative_to(SHARDS_DIR.parent).as_posix(),
            )
            written += 1
    archive.writestr("README.txt", README)

print(f"zip: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:,.1f} MB, {written} files)")


## 8. Kiểm tra batch

Chỉ verify shape/count của từng shard; FAISS tổng được build ở local bằng
`uv run ir import-shards` (hoặc `uv run ir merge`).

In [ ]:
total_keyframes = 0
for summary in shard_summaries:
    shard_dir = SHARDS_DIR / summary["video_id"]
    manifest = json.loads((shard_dir / "manifest.json").read_text(encoding="utf-8"))
    embeddings = np.load(shard_dir / "embeddings.f16.npy", mmap_mode="r")
    ids = np.load(shard_dir / "id_map.npy")
    count = int(manifest["count"])
    assert embeddings.shape == (count, DIM), (summary["video_id"], embeddings.shape)
    assert ids.shape == (count,), (summary["video_id"], ids.shape)
    total_keyframes += count
    print(f"{summary['video_id']}: {count:,} keyframes")

print(f"batch: {len(shard_summaries):,} videos, {total_keyframes:,} keyframes")
if not VIDEO_IDS and VIDEO_LIMIT > 0:
    print("next EMBED_VIDEO_OFFSET:", VIDEO_OFFSET + len(selected))


## Sau khi chạy

1. Tải `artifacts_embed_shards.zip`.
2. Ở local chạy `uv run ir import-shards artifacts_embed_shards.zip`.
   Lệnh này giải nén vào `state/embed/shards/` rồi build luôn
   `embeddings.f16.npy`, `id_map.npy`, `faiss.index`, `manifest.json`.
3. Có thể import/merge sau mỗi batch; `ir merge` luôn đọc lại tất cả shard.
